# Roman Motoru — Boş bir model bir roman okursa ne olur?

Bu defterde **hiçbir şey bilmeyen** bir model alıp ona tek bir roman okutacağız:
Sabahattin Ali — *Kürk Mantolu Madonna* (1943, kamu malı).

Model tek bir şey yapacak: **bir sonraki harfi tahmin etmek.**

**Nasıl kullanılır:** Hücreleri yukarıdan aşağıya ▶ düğmesiyle sırayla çalıştırın.
Deneyleri yalnız **4 numaralı hücrede** yapacaksınız: ayarı değiştirin, o hücrede tekrar ▶.

**Bu tarayıcı sürümüdür (hesap gerekmez).** Kaydırıcı yoktur: 4. hücredeki sayıları (`BAGLAM = 4` gibi)
elle değiştirin, sonra ▶. İlk ▶ Python'un tarayıcıya yüklenmesi için ~10 saniye sürer.
Değişiklikleriniz yalnız bu tarayıcıda kalır.

In [ ]:
#@title 1 · Romanı indir (bir kez çalıştırın)
import os, re, json, html, random, urllib.request, urllib.parse
from collections import Counter, defaultdict

def romani_temizle(ham_html):
    t = re.sub(r"<br\s*/?>|</p>", "\n", ham_html)
    t = html.unescape(re.sub(r"<[^>]+>", "", t)).replace("­", "")
    bas, son = t.find("Simdiye kadar"), t.find("Bu eser, kültürel")
    return re.sub(r"\s+", " ", t[bas:son]).strip()

if os.path.exists("roman.txt"):            # internet yoksa: dosyayı sol paneldeki Dosyalar'a sürükleyin
    metin = open("roman.txt", encoding="utf-8").read()
else:
    adres = ("https://tr.wikisource.org/w/api.php?action=parse&prop=text&format=json&formatversion=2&page="
             + urllib.parse.quote("Kürk Mantolu Madonna"))
    istek = urllib.request.Request(adres, headers={"User-Agent": "roman-motoru-ders/1.0"})
    try:
        metin = romani_temizle(json.load(urllib.request.urlopen(istek, timeout=30))["parse"]["text"])
    except Exception as hata:
        raise SystemExit(f"Roman internetten indirilemedi ({type(hata).__name__}).\n"
                         "Çözüm: öğretmeninizin verdiği roman.txt dosyasını bu defterle AYNI klasöre koyun "
                         "(Colab'da: soldaki klasör simgesi → dosyayı sürükleyin) ve bu hücreyi tekrar çalıştırın.")

print(f"Roman hazır: {len(metin):,} harf".replace(",", "."))
print("İlk satırlar:", metin[:200], "...")

## 2 · Hiç okumamış model

Romanı henüz okumamış bir model, bildiği harflerin hepsini **eşit olasılıkla** seçer.

In [ ]:
#@title 2 · Boş modelin yazısı
harfler = sorted(set(metin))
random.seed(1)
print(f"Modelin bildiği {len(harfler)} farklı harf ve işaret var. Hepsi eşit olasılıkta:\n")
print("".join(random.choice(harfler) for _ in range(300)))

In [ ]:
#@title 3 · Motor (çalıştırın, değiştirmeyin)
def romani_oku(baglam):
    """Romanı baştan sona okur: her 'baglam' harflik dizinin ardından hangi harf kaç kez gelmiş, sayar."""
    sayac = defaultdict(Counter)
    for i in range(len(metin) - baglam):
        sayac[metin[i:i + baglam]][metin[i + baglam]] += 1
    return sayac

def olasiliklar(sayac, onceki, baglam, sicaklik):
    """Son 'baglam' harfe bakar, bir sonraki harfin olasılıklarını verir. Görülmemişse None."""
    say = sayac.get(onceki[-baglam:])
    if not say:
        return None
    agirlik = {h: n ** (1 / sicaklik) for h, n in say.items()}   # sıcaklık: düşükse en olası öne çıkar
    toplam = sum(agirlik.values())
    return {h: a / toplam for h, a in agirlik.items()}

def yaz(sayac, baslangic, baglam, sicaklik, uzunluk, tohum):
    random.seed(tohum)
    yazi, tikanma = baslangic, 0
    for _ in range(uzunluk):
        p = olasiliklar(sayac, yazi, baglam, sicaklik)
        if p is None:                       # bu dizi romanda hiç geçmemiş: rastgele bir harfle devam
            yazi += random.choice(metin); tikanma += 1
        else:
            yazi += random.choices(list(p), weights=list(p.values()))[0]
    return yazi, tikanma

def ezber_orani(uretilen, parca=20):
    """Üretilen yazının 20 harflik parçalarından kaçı romanda AYNEN geçiyor?"""
    adimlar = range(0, max(0, len(uretilen) - parca), 5)
    if not adimlar:
        return 0.0
    return 100 * sum(uretilen[i:i + parca] in metin for i in adimlar) / len(adimlar)

print("Motor hazır.")

In [ ]:
#@title 4 · DENEY — ayarları değiştirin, sonra bu hücrede ▶
BAGLAM = 4              #@param {type:"slider", min:1, max:10, step:1}
SICAKLIK = 1.0          #@param {type:"slider", min:0.1, max:2.0, step:0.1}
UZUNLUK = 300           #@param {type:"slider", min:50, max:1000, step:50}
BASLANGIC = "Raif Efendi"  #@param {type:"string"}
TOHUM = 42              #@param {type:"integer"}

sayac = romani_oku(BAGLAM)
print(f"Model romanı okudu: {len(sayac):,} farklı {BAGLAM} harflik dizi saydı.\n".replace(",", "."))

p = olasiliklar(sayac, BASLANGIC, BAGLAM, SICAKLIK)
print(f'"{BASLANGIC}" yazısından sonra hangi harf gelir? (model yalnız son {BAGLAM} harfe bakıyor: "{BASLANGIC[-BAGLAM:]}")')
if p is None:
    print("  Bu harf dizisi romanda hiç geçmiyor. BASLANGIC'ı değiştirin ya da BAGLAM'ı küçültün.")
else:
    for h, o in sorted(p.items(), key=lambda x: -x[1])[:8]:
        print(f"  {repr(h):6} %{o * 100:5.1f}  " + "█" * round(o * 40))

yazi, tikanma = yaz(sayac, BASLANGIC, BAGLAM, SICAKLIK, UZUNLUK, TOHUM)
print("\n--- MODELİN YAZISI ---\n" + yazi + "\n----------------------")
print(f"Romanda aynen geçen parça oranı: %{ezber_orani(yazi[len(BASLANGIC):]):.0f}"
      + (f"   (tıkanma: {tikanma} kez)" if tikanma else ""))

## Deney kartı

| # | Değiştirin | Gözleyin |
|---|---|---|
| 1 | `BAGLAM` 1 → 3 → 6 | Harf çorbası ne zaman kelimeye, ne zaman cümleye dönüşüyor? |
| 2 | `SICAKLIK` 0.2 ↔ 1.8 | Düşükte ne tekrarlanıyor? Yüksekte ne bozuluyor? |
| 3 | `BAGLAM` 10 | "Romanda aynen geçen" oranı kaç? Model yazıyor mu, kopyalıyor mu? |
| 4 | `BASLANGIC` kendi cümleniz | Romanda hiç geçmeyen bir başlangıçta ne oluyor? |

**Asıl soru:** Akıcı yazmak, anlamak mıdır?